In [1]:
# I am helping a food truck owner run a tiny order system. I'll help clean menu text, manage orders, and print a receipt.

# --- Data---
# Menu master data (immutable on purpose
MENU = (
    ("americano", 3.25),
    ("latte", 4.50),
    ("mocha", 4.75),
    ("chai", 3.95),
    ("espresso", 2.75)
)

# Raw marketing string (Messy spacing and casing)
marketing_banner = "  try our NEW   LATTE  and   Mocha! best   in town  "

# Orders coming from a basic kiosk (item names only, includes extra spaces/case variants)
raw_orders = ["Latte", " AMERICANO ", "chai", "Mocha", "espresso", "latte", "chai", "LATTE "]



In [2]:
#1 Clean and normalize the banner text 
banner_clean = " ".join(marketing_banner.strip().split()).title()#title() capitalizes the first letter of each word
print("Before:", repr(marketing_banner)) #repr shows hidden characters like spaces
print("After :", repr(banner_clean))

Before: '  try our NEW   LATTE  and   Mocha! best   in town  '
After : 'Try Our New Latte And Mocha! Best In Town'


In [3]:
#2 Check if the banner mentions 'Latte' 
has_latte = "Latte" in banner_clean #True if 'Latte' is mentioned in string sentence
print(f"Latte highlighted? {has_latte}") #f-string for formatted output

Latte highlighted? True


In [4]:
# 3) Replace 'Mocha' with 'Mocha (chocolate)'
banner_with_details = banner_clean.replace("Mocha", "Mocha (Chocolate)")
print(banner_with_details)

Try Our New Latte And Mocha (Chocolate)! Best In Town


In [5]:
# 4) Print first 12 and last 10 characters of the cleaned banner
first_12 = banner_clean[:12] #First 12 characters of cleaned banner
last_10  = banner_clean[-10:] #Last 10 characters of cleaned banne
print(f"{first_12}...{last_10}")

Try Our New ...st In Town


In [6]:
# 5) Loop through MENU to print items with prices
for name, price in MENU:
    print(f"{name} — ${price:.2f}")#.2f for 2 decimal places

americano — $3.25
latte — $4.50
mocha — $4.75
chai — $3.95
espresso — $2.75


In [7]:
# 6) Unpack MENU into item_names and item_prices lists
item_names, item_prices = list(zip(*MENU))  # tuple of tuples -> two tuples #zip(*Menu) unzips the MENU #item_names, item_prices are tuples
item_names = list(item_names) #convert tuples to lists
item_prices = list(item_prices)
print(item_names)
print(item_prices)

['americano', 'latte', 'mocha', 'chai', 'espresso']
[3.25, 4.5, 4.75, 3.95, 2.75]


In [8]:
# 7) Trying to change espresso price directly in MENU (should fail), then rebuilding MENU_UPDATED
try:
    # This raises a TypeError because tuples are immutable
    MENU[4] = ("espresso", 2.95)  # type: ignore
except TypeError as e:
    print("Tuples are immutable:", type(e).__name__)

# Correct approach: rebuild a new tuple with the updated price
MENU_UPDATED = tuple((n, 2.95) if n == "espresso" else (n, p) for n, p in MENU) #update espresso price
print(MENU_UPDATED)

Tuples are immutable: TypeError
(('americano', 3.25), ('latte', 4.5), ('mocha', 4.75), ('chai', 3.95), ('espresso', 2.95))


In [9]:
# 8) Cleaning the order names (trim and lowercase)
orders = [o.strip().lower() for o in raw_orders]
print("Before:", raw_orders)
print("After :", orders)

Before: ['Latte', ' AMERICANO ', 'chai', 'Mocha', 'espresso', 'latte', 'chai', 'LATTE ']
After : ['latte', 'americano', 'chai', 'mocha', 'espresso', 'latte', 'chai', 'latte']


In [10]:
# 9) Count specific items
lattes = orders.count("latte")
chais  = orders.count("chai")
print(f"Lattes: {lattes} | Chais: {chais}")

Lattes: 3 | Chais: 2


In [11]:
# 10) Validating orders against menu items
valid_names = set(n for n, x in MENU_UPDATED) #this creates a set of valid item names from the updated menu, x is a placeholder for price,n is item name
print("Valid items:", valid_names)
invalid = [o for o in orders if o not in valid_names]
print("Invalid items:", invalid)  # should be [] if cleaned correctly

Valid items: {'chai', 'americano', 'mocha', 'espresso', 'latte'}
Invalid items: []


In [12]:
# 11) Sort and create a unique list preserving first occurrence order
orders_sorted = sorted(orders)#sorted() returns a new sorted list from the items in iterable
unique_order = []
for item in orders:
    if item not in unique_order:
        unique_order.append(item)

print("Sorted orders:", orders_sorted)
print("Unique items (first-seen order):", unique_order)

Sorted orders: ['americano', 'chai', 'chai', 'espresso', 'latte', 'latte', 'latte', 'mocha']
Unique items (first-seen order): ['latte', 'americano', 'chai', 'mocha', 'espresso']


In [13]:
# 12) Create basket_counts as list of tuples (item, count), sorted by item name
basket_counts = sorted([(u, orders.count(u)) for u in unique_order],key=lambda x: x[0]) # x:x[0] sorts by item name, x:x[1] would sort by count
basket_counts

[('americano', 1), ('chai', 2), ('espresso', 1), ('latte', 3), ('mocha', 1)]

In [14]:
# 13) Compute subtotal from basket_counts and MENU_UPDATED
price_map = dict(MENU_UPDATED)

print("Price map:", price_map)
print("Basket counts:", basket_counts)
subtotal = sum(price_map[item] * qty for item, qty in basket_counts) #price_map[item] gets price from dict
print(f"Subtotal = ${subtotal:.2f}")

Price map: {'americano': 3.25, 'latte': 4.5, 'mocha': 4.75, 'chai': 3.95, 'espresso': 2.95}
Basket counts: [('americano', 1), ('chai', 2), ('espresso', 1), ('latte', 3), ('mocha', 1)]
Subtotal = $32.35


In [15]:
# 14) Add 8.25% tax and compute grand total
tax_rate = 0.0825
tax_amount = subtotal * tax_rate
grand_total = subtotal + tax_amount
print(f"Tax (8.25%) = ${tax_amount:.2f}")
print(f"Grand Total  = ${grand_total:.2f}")

Tax (8.25%) = $2.67
Grand Total  = $35.02


In [16]:
# 15) Find the cheapest and costliest items
cheapest = min(MENU_UPDATED, key=lambda x: x[1])#find tuple with min price

costliest = max(MENU_UPDATED, key=lambda x: x[1])#find tuple with max price 
print(f"Cheapest: {cheapest[0]} (${cheapest[1]:.2f}) | Costliest: {costliest[0]} (${costliest[1]:.2f})")

Cheapest: espresso ($2.95) | Costliest: mocha ($4.75)


In [17]:
# 16) Print a formatted receipt
print("---- Coffee Cart ----")
print("Items:")
for item, qty in basket_counts:
    line_total = price_map[item] * qty
    print(f"  {item.ljust(9)} x {str(qty).rjust(2)} = ${line_total:6.2f}")#ljust will left align item name in 9 spaces, rjust will right align qty in 2 spaces
print(f"{'Subtotal'.ljust(16)} ${subtotal:6.2f}")
print(f"{'Tax (8.25%)'.ljust(16)} ${tax_amount:6.2f}")
print(f"{'TOTAL'.ljust(16)} ${grand_total:6.2f}")
print("----------------------")

---- Coffee Cart ----
Items:
  americano x  1 = $  3.25
  chai      x  2 = $  7.90
  espresso  x  1 = $  2.95
  latte     x  3 = $ 13.50
  mocha     x  1 = $  4.75
Subtotal         $ 32.35
Tax (8.25%)      $  2.67
TOTAL            $ 35.02
----------------------


In [18]:
# 17) Personalized message with f-string
customer_name = "Alex"
print(f"Thanks, {customer_name}! Your total is ${grand_total:.2f}. Have a great day!")

Thanks, Alex! Your total is $35.02. Have a great day!


In [19]:
# 18) Promo code SAVE5 -> $0.05 discount per item
promo = "SAVE5"
discount_per_item = 0.05 if promo == "SAVE5" else 0.0
total_items = sum(qty for _, qty in basket_counts)
discount = discount_per_item * total_items
grand_total_after_discount = max(0, grand_total - discount)#max ensures total doesn't go below 0
print(f"Items: {total_items} | Discount: ${discount:.2f} | New Total: ${grand_total_after_discount:.2f}")

Items: 8 | Discount: $0.40 | New Total: $34.62


In [20]:
# 19) Top 2 sellers by quantity
top2 = sorted(basket_counts, key=lambda x: x[1], reverse=True)[:2]#reverse=True sorts in descending order, [:2] gets the top 2
print("Top 2 sellers:", top2)

Top 2 sellers: [('latte', 3), ('chai', 2)]


In [21]:
# 20) Menu search (substring)
query = "es"
matches = [n for n, _ in MENU_UPDATED if query.lower() in n.lower()]
print(f"Search '{query}':", matches)

Search 'es': ['espresso']


In [22]:
# 21) Immutable daily menu snapshot tuple of strings "name:$price"
daily_menu_snapshot = tuple(f"{n}:${p:.2f}" for n, p in MENU_UPDATED)
daily_menu_snapshot

('americano:$3.25',
 'latte:$4.50',
 'mocha:$4.75',
 'chai:$3.95',
 'espresso:$2.95')